# FaceProof 05 — multi-generator evaluation (Day 5)

Add ≥2 more **unseen** generators so RQ1 isn't a single SD data point. We use **SFHQ-T2I**
(SDXL / Flux / DALL-E 3 faces, labelled in a CSV). Pipeline: download → compression-match
(same `src.preprocessing`) → cache features → evaluate the **saved C1/C4 probes** per generator.

> The CSV schema varies — the notebook prints columns so you confirm the image + generator
> columns before the per-generator loop.

## 1. Setup (GPU on) + Kaggle

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "day1-preprocess-notebook"   # change to "main" after merge
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q kaggle open_clip_torch scikit-learn joblib pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
RESULT_FIELDS = ['condition','train_gen','test_gen','corruption','strength','seed','metric','value']
def log_result(**row):
    full = {k: row.get(k, '') for k in RESULT_FIELDS}
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if new: w.writeheader()
        w.writerow(full)

In [ ]:
from google.colab import files
print('Upload kaggle.json:'); files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# Verify the slug on Kaggle if this 404s.
!kaggle datasets download -d selfishgene/sfhq-t2i-synthetic-faces-from-text-2-image-models -p /content/t2i --unzip

## 2. Find the images + the labels CSV

In [ ]:
T2I = Path('/content/t2i')
csvs = list(T2I.rglob('*.csv'))
print('CSV files:', csvs)
meta = pd.read_csv(csvs[0])
print('columns:', list(meta.columns)); meta.head()

## 3. Map images → generator (CONFIRM column names from the printout above)

In [ ]:
# >>> EDIT these two to match the printed columns <<<
COL_IMAGE = 'image_filename'      # column holding the file name
COL_GEN   = 'model'               # column holding the generator name (SDXL/Flux/DALL-E ...)

img_root = csvs[0].parent
def find_img(name):
    hits = list(img_root.rglob(name))
    return str(hits[0]) if hits else None
meta['path'] = meta[COL_IMAGE].map(find_img)
meta = meta.dropna(subset=['path'])
assert len(meta) > 0, f"No images matched; check COL_IMAGE='{COL_IMAGE}'"
print(meta[COL_GEN].value_counts())

## 4. Preprocess + cache features per generator

In [ ]:
import yaml, joblib
from src.preprocessing import preprocess_image
from src.features import extract_clip, extract_resnet
from PIL import Image

dcfg = yaml.safe_load(open('configs/data.yaml')); IMG=dcfg['image_size']; Q=dcfg['jpeg_match_quality']
mcfg = yaml.safe_load(open('configs/model.yaml'))['clip']
probes = {c: joblib.load(PROBES_ROOT / f'{c}.joblib') for c in ['c1_resnet','c4_clip']}

# Also need REAL features to make each eval real-vs-fake. Reuse cached clean real crops.
# Held-out reals only (label 0 in test_indist) -> no training images leak into eval.
_mani = pd.read_csv(MANIFEST)
real_paths = _mani[(_mani.label==0) & (_mani['split']=='test_indist')]['path'].tolist()[:3000]
assert real_paths, 'no held-out reals found in manifest'
import shutil
def crops_for(paths, tag):
    out = Path(f'/content/_crops_{tag}'); 
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    kept=[]
    for i,p in enumerate(paths):
        im = preprocess_image(p, IMG, Q, detector=None)
        if im is None: continue
        op = out/f'{i:06d}.jpg'; im.save(op,'JPEG',quality=Q); kept.append(str(op))
    return kept

## 5. Evaluate saved probes on each unseen generator (vs real)

In [ ]:
from src.metrics import auroc, eer
from src.probe import predict_proba

# Real side: features from cached real crops.
Xr_clip = extract_clip(real_paths, backbone=mcfg['backbone'], pretrained=mcfg['pretrained'], batch_size=64, cache=None)
Xr_res  = extract_resnet(real_paths, batch_size=64, cache=None)

rows=[]
for gen, grp in meta.groupby(COL_GEN):
    fake_paths = crops_for(grp['path'].tolist()[:3000], gen.replace('/','_'))
    Xf_clip = extract_clip(fake_paths, backbone=mcfg['backbone'], pretrained=mcfg['pretrained'], batch_size=64, cache=None)
    Xf_res  = extract_resnet(fake_paths, batch_size=64, cache=None)
    feats = {'c4_clip': (Xr_clip, Xf_clip), 'c1_resnet': (Xr_res, Xf_res)}
    for cond,(Xr,Xf) in feats.items():
        Xall=np.vstack([Xr,Xf]); yall=np.r_[np.zeros(len(Xr)),np.ones(len(Xf))]
        p=predict_proba(probes[cond],Xall); au=auroc(yall,p); er=eer(yall,p)
        log_result(condition=cond, train_gen='stylegan', test_gen=str(gen), corruption='none',
                   seed=13, metric='AUROC', value=round(au,4))
        log_result(condition=cond, train_gen='stylegan', test_gen=str(gen), corruption='none',
                   seed=13, metric='EER', value=round(er,4))
        rows.append({'condition':cond,'generator':gen,'AUROC':round(au,4),'EER':round(er,4)})
    print('done', gen)
print(pd.DataFrame(rows).to_string(index=False))
print('✅ Day 5 gate: ≥2 unseen generators in results.csv')